# BookLM: проверяем контракт до реализации

**Вопрос.** Сколько параметров и памяти должна иметь сквозная модель книги? Сначала прочитайте конфигурацию и попробуйте получить итог вручную. Затем выполните ячейки сверху вниз.

In [ ]:
from dataclasses import dataclass, asdict
import hashlib
import json
import sys

@dataclass(frozen=True)
class BookLMConfig:
    vocab_size: int = 8192
    d_model: int = 256
    n_layers: int = 6
    n_heads: int = 8
    n_kv_heads: int = 2
    d_ff: int = 768
    max_seq_len: int = 512
    tie_embeddings: bool = True

cfg = BookLMConfig()
assert cfg.d_model % cfg.n_heads == 0
assert cfg.n_heads % cfg.n_kv_heads == 0
print(sys.version.split()[0], asdict(cfg))

## Число параметров

Выходная матрица использует ту же память, что таблица токенов. В линейных слоях нет смещений. У каждой RMSNorm один обучаемый масштаб на канал.

In [ ]:
head_dim = cfg.d_model // cfg.n_heads
kv_width = cfg.n_kv_heads * head_dim

embedding = cfg.vocab_size * cfg.d_model
attention = (
    cfg.d_model * cfg.d_model
    + 2 * cfg.d_model * kv_width
    + cfg.d_model * cfg.d_model
)
swiglu = 3 * cfg.d_model * cfg.d_ff
norms_per_block = 2 * cfg.d_model
final_norm = cfg.d_model
total = embedding + cfg.n_layers * (attention + swiglu + norms_per_block) + final_norm

parts = {
    'embedding': embedding,
    'attention_per_block': attention,
    'swiglu_per_block': swiglu,
    'norms_per_block': norms_per_block,
    'final_norm': final_norm,
    'total': total,
}
assert total == 6_622_464
parts

## Память параметров и KV-кэша

Здесь считаются именно байты логических данных. Аллокатор может округлить физическое выделение, а среда выполнения добавит рабочие области.

In [ ]:
bf16_bytes = 2
parameter_bytes = total * bf16_bytes
kv_bytes_per_token_per_layer = 2 * cfg.n_kv_heads * head_dim * bf16_bytes
kv_bytes_per_token = cfg.n_layers * kv_bytes_per_token_per_layer
kv_bytes_full_context = cfg.max_seq_len * kv_bytes_per_token

assert parameter_bytes == 13_244_928
assert kv_bytes_per_token_per_layer == 256
assert kv_bytes_per_token == 1_536
assert kv_bytes_full_context == 786_432
print(f'parameters BF16: {parameter_bytes / 2**20:.2f} MiB')
print(f'KV, one 512-token request: {kv_bytes_full_context / 2**10:.0f} KiB')

In [ ]:
canonical_config = json.dumps(asdict(cfg), sort_keys=True, separators=(',', ':')).encode()
config_sha256 = hashlib.sha256(canonical_config).hexdigest()
manifest = {
    'id': 'booklm-base-config-v1',
    'parents': {},                     # конфигурация --- корень: родителей нет
    'parameter_count': total,
    'config_sha256': config_sha256,
}
assert len(config_sha256) == 64
manifest

## Задания

1. До запуска предскажите, как изменится число параметров при отвязывании выходной матрицы.
2. Поставьте `n_kv_heads=8` и объясните, почему параметры внимания и KV-кэш выросли по-разному.
3. Добавьте расчёт памяти AdamW при FP32-состояниях. Явно перечислите учтённые копии параметров.